# 2장 1강: t-검정의 이론과 가정 — 실습문제

## 실습 목표

- 평균 차이와 표준오차를 이용해 t통계량을 직접 계산할 수 있다.
- 단일표본 t검정으로 하나의 표본평균과 기준값을 비교할 수 있다.
- 독립성·정규성·등분산성을 확인한 뒤 독립표본 t검정을 수행할 수 있다.
- 독립표본과 대응표본 상황을 구분하고 적절한 함수를 선택할 수 있다.
- p-value를 유의수준과 비교하여 검정 결과를 올바르게 해석할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `KitchenQual` | 주방 품질 |

> 모든 검정은 양측검정이며 유의수준 `α = 0.05`를 사용합니다.  
> Ames 표본 추출에는 지정된 `random_state`를 사용해 결과를 재현합니다.

In [6]:
# T-통계량 : 관찰된 평균 차이가 표준오차의 몇 배인지를 나타내는 검증용 숫자
# -> 평균이 75점, 기준이 70점, 표준오차가 약 0.57점이면 t값은 약 8.6이고
# 평균의 차이가 표준오차의 약 8.6배정도 라는 뜻

# 독립표본 t검정 : 서로 독립된 두 집단의 모평균이 같은지 비교하는 검정
# -> 광고 A와 B를 본 서로 다른 고객의 평균 구매 금액을 비교함
# welch t검정 : 두 집단의 분산이 같다는 가정 x, 독립된 두 집단의 평균을 비교

# shapiro(정규성검증) : 데이터가 정규분포에서 왔다는 귀무가설을 검토하는 정규성 검정
# -> p값이 0.05보다 작으면 정규성 가정과 맞지 않는 근거가 있다고 해석

# Levene(등분산성) : 여러 집단의 분산이 같다는 귀무가설을 검토하는 검정
# -> 두 집단 비교에서 p값이 0.05보다 작으면 등분산 가정과 맞지않는 근거가 있다고 해석


## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [2]:
# 실습 준비 코드를 작성하세요.

# 1. 필요한 라이브러리 불러오기
import pandas as pd
from scipy import stats

# 2. Ames Housing 데이터를 df에 불러오기
df = pd.read_csv('ames_housing.csv')

print(df.isna().sum())

# 3. 데이터의 행과 열 개수, 컬럼명, 상위 5개 행 확인
print("행, 열 개수:", df.shape)
print()
print("컬럼명:", df.columns.tolist())
print()
print("상위 5개 행:")
print(df.head())

SalePrice       0
GrLivArea       0
LotArea         0
OverallQual     0
KitchenQual     0
CentralAir      0
HeatingQC       0
PavedDrive      0
Neighborhood    0
YearBuilt       0
dtype: int64
행, 열 개수: (1460, 10)

컬럼명: ['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual', 'CentralAir', 'HeatingQC', 'PavedDrive', 'Neighborhood', 'YearBuilt']

상위 5개 행:
   SalePrice  GrLivArea  LotArea  OverallQual KitchenQual CentralAir  \
0     208500       1710     8450            7          Gd          Y   
1     181500       1262     9600            6          TA          Y   
2     223500       1786    11250            7          Gd          Y   
3     140000       1717     9550            7          Gd          Y   
4     250000       2198    14260            8          Gd          Y   

  HeatingQC PavedDrive Neighborhood  YearBuilt  
0        Ex          Y      CollgCr       2003  
1        Ex          Y      Veenker       1976  
2        Ex          Y      CollgCr       2001  
3   

---

## 필수 1. t통계량 직접 계산과 단일표본 t검정

### 문제 1-1. 평균 판매가격은 180,000달러와 다른가?

#### 문제 설명

Ames 주택 30개를 표본으로 추출하여 평균 판매가격이 기준값 180,000달러와 다른지 확인합니다. 먼저 t통계량을 직접 계산한 뒤 `ttest_1samp()` 결과와 비교하세요.

#### 요구사항

1. `SalePrice`에서 `n=30`, `random_state=2`로 표본을 추출하여 `sale_sample`에 저장하세요.
2. 표본 수, 표본평균, 표본표준편차를 출력하세요.
3. `stats.sem()`을 이용해 표준오차를 계산하세요.
4. 다음 식으로 t통계량을 직접 계산하세요.  
   `t = (표본평균 - 기준값) / 표준오차`
5. Shapiro-Wilk 검정으로 표본의 정규성을 확인하세요.
6. 다음 가설을 작성하세요.
   - H₀: 모집단 평균 판매가격은 180,000달러이다.
   - H₁: 모집단 평균 판매가격은 180,000달러가 아니다.
7. `stats.ttest_1samp()`로 단일표본 t검정을 수행하세요.
8. 직접 계산한 t통계량과 함수가 반환한 t통계량을 비교하세요.
9. p-value를 이용해 귀무가설 기각 여부를 판단하세요.

#### 해석 질문

**Q1.** t통계량은 평균 차이와 표준오차를 어떻게 이용한 값인가요?  
**Q2.** 같은 평균 차이라면 표준오차가 작아질수록 t통계량의 절댓값은 어떻게 변하나요?  
**Q3.** 직접 계산한 t통계량과 `ttest_1samp()`의 t통계량은 일치하나요?  
**Q4.** 검정 결과 평균 판매가격이 180,000달러와 다르다고 판단할 수 있나요?

#### 제출 결과

- 기술통계량과 정규성 결과
- 직접 계산한 표준오차와 t통계량
- 단일표본 t검정 결과
- 귀무가설 판단과 해석
- Q1~Q4 답변

In [2]:
# 필수 1 코드를 작성하세요.

import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1. 표본 추출
sale_sample = df['SalePrice'].sample(n=30, random_state=2)

# 2. 표본 수, 표본평균, 표본표준편차 출력
n = len(sale_sample)
sample_mean = sale_sample.mean()
sample_std = sale_sample.std(ddof=1)  # 표본표준편차 (ddof=1)
print(f"표본 수: {n}")
print(f"표본평균: {sample_mean:.4f}")
print(f"표본표준편차: {sample_std:.4f}")

# 3. 표준오차 계산
se = stats.sem(sale_sample)
print(f"\n표준오차(SE): {se:.4f}")

# 4. t통계량 직접 계산
popmean = 180000
t_manual = (sample_mean - popmean) / se
print(f"\n직접 계산한 t통계량: {t_manual:.4f}")

# 5. 정규성 검정 (Shapiro-Wilk)
stat_shapiro, p_shapiro = stats.shapiro(sale_sample)
print(f"\nShapiro-Wilk: 통계량={stat_shapiro:.4f}, p-value={p_shapiro:.4f}")
print(f"정규성 충족(p>=0.05): {p_shapiro >= 0.05}")

# 6. 가설
print("\nH0: 모집단 평균 판매가격은 180,000달러이다.")
print("H1: 모집단 평균 판매가격은 180,000달러가 아니다.")

# 7. scipy 함수로 단일표본 t검정
t_stat, p_value = stats.ttest_1samp(sale_sample, popmean=popmean)
print(f"\nttest_1samp 결과: t통계량={t_stat:.4f}, p-value={p_value:.4f}")

# 8. 직접 계산한 값과 함수 반환값 비교
print(f"\n직접 계산 t: {t_manual:.4f}")
print(f"함수 반환 t: {t_stat:.4f}")
print(f"두 값이 (거의) 일치하는가: {np.isclose(t_manual, t_stat)}")

# 9. 결론
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.4f}) < alpha({alpha}) → 귀무가설 기각")
    print("모집단 평균 판매가격은 180,000달러와 다르다고 할 수 있습니다.")
else:
    print(f"\np-value({p_value:.4f}) >= alpha({alpha}) → 귀무가설 기각 실패")
    print("모집단 평균 판매가격이 180,000달러와 다르다고 할 근거가 부족합니다.")

표본 수: 30
표본평균: 233323.8000
표본표준편차: 86013.0117

표준오차(SE): 15703.7556

직접 계산한 t통계량: 3.3956

Shapiro-Wilk: 통계량=0.9538, p-value=0.2133
정규성 충족(p>=0.05): True

H0: 모집단 평균 판매가격은 180,000달러이다.
H1: 모집단 평균 판매가격은 180,000달러가 아니다.

ttest_1samp 결과: t통계량=3.3956, p-value=0.0020

직접 계산 t: 3.3956
함수 반환 t: 3.3956
두 값이 (거의) 일치하는가: True

p-value(0.0020) < alpha(0.05) → 귀무가설 기각
모집단 평균 판매가격은 180,000달러와 다르다고 할 수 있습니다.


### 필수 1 답변 작성란

**Q1.** t통계량은 평균 차이와 표준오차를 어떻게 이용한 값인가요?  
-> 관측된 평균과 기준값의 차이를 (그 차이의 불확실성을 의미하는) 표준오차로 나눈 값

**Q2.** 같은 평균 차이라면 표준오차가 작아질수록 t통계량의 절댓값은 어떻게 변하나요?  
-> 분모(표준오차)가 작아지므로 t통계량의 절대값은 커짐

**Q3.** 직접 계산한 t통계량과 `ttest_1samp()`의 t통계량은 일치하나요?  
-> 일치함. 두 방법 모두 약 3.3956으로 계산됨

**Q4.** 검정 결과 평균 판매가격이 180,000달러와 다르다고 판단할 수 있나요?
->  그렇다. p-value가 약 0.002로 0.05보다 작아 귀무가설을 기각함
-> 따라서 평균 판매가격은 180,000달러와 다름

---

## 필수 2. 독립표본 t검정의 가정 점검과 수행

### 문제 2-1. 주방 품질 `Gd`와 `TA` 집단의 평균 판매가격 비교

#### 문제 설명

주방 품질이 `Gd`인 주택과 `TA`인 주택에서 각각 20개를 추출해 평균 판매가격을 비교합니다. 두 집단은 서로 다른 주택으로 구성되어 있습니다.

#### 요구사항

1. `KitchenQual == "Gd"`와 `KitchenQual == "TA"` 집단의 `SalePrice`에서 각각 `n=20`, `random_state=42`로 표본을 추출하세요.
2. 두 집단의 표본 수와 평균을 확인하세요.
3. 데이터 수집 구조를 근거로 두 집단의 독립성을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 가정 점검 결과에 따라 `equal_var=True` 또는 `equal_var=False`를 결정하세요.
7. `stats.ttest_ind()`로 독립표본 t검정을 수행하세요.
8. t통계량과 p-value를 출력하고 두 집단 평균 차이를 해석하세요.

#### 해석 질문

**Q1.** 두 집단이 독립표본인 이유는 무엇인가요?  
**Q2.** 독립성은 별도의 p-value로 확인할 수 있나요?  
**Q3.** 정규성과 등분산성 가정은 각각 충족되나요?  
**Q4.** `equal_var`에는 어떤 값을 사용해야 하나요?  
**Q5.** 두 집단의 평균 판매가격에는 통계적으로 유의한 차이가 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 독립성 설명
- 정규성 및 등분산성 검정 결과
- `equal_var` 선택 근거
- 독립표본 t검정 결과와 해석
- Q1~Q5 답변

In [4]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1. 표본 추출 (n=20, random_state=42)
group_gd = df.loc[df['KitchenQual'] == 'Gd', 'SalePrice'].sample(n=20, random_state=42)
group_ta = df.loc[df['KitchenQual'] == 'TA', 'SalePrice'].sample(n=20, random_state=42)

# 2. 표본 수와 평균 확인
print(f"group_gd 표본 수: {len(group_gd)}, 평균: {group_gd.mean():.2f}")
print(f"group_ta 표본 수: {len(group_ta)}, 평균: {group_ta.mean():.2f}")

# 3. 독립성 설명
print("\ngroup_gd와 group_ta는 KitchenQual 값이 다른 서로 별개의 주택(행)에서 추출되었습니다.")
print("한 주택이 Gd이면서 동시에 TA일 수 없고, 두 표본 사이에 짝지어진 관계도 없으므로 두 집단은 독립입니다.")

# 4. 정규성 검정
stat_gd, p_gd = stats.shapiro(group_gd)
stat_ta, p_ta = stats.shapiro(group_ta)
print(f"\nShapiro-Wilk (group_gd): 통계량={stat_gd:.4f}, p-value={p_gd:.4f}")
print(f"Shapiro-Wilk (group_ta): 통계량={stat_ta:.4f}, p-value={p_ta:.4f}")

# 5. 등분산 검정
stat_levene, p_levene = stats.levene(group_gd, group_ta)
print(f"\nLevene 등분산 검정: 통계량={stat_levene:.4f}, p-value={p_levene:.4f}")

# 6. equal_var 결정
equal_var = p_levene >= 0.05
print(f"\n등분산성 충족(p>=0.05): {equal_var} -> equal_var={equal_var}")

# 7. 독립표본 t검정
t_stat, p_value = stats.ttest_ind(group_gd, group_ta, equal_var=equal_var)
print(f"\nt통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

# 8. 해석
alpha = 0.05
mean_diff = group_gd.mean() - group_ta.mean()
if p_value < alpha:
    print(f"\np-value({p_value:.4f}) < alpha({alpha}) → 귀무가설 기각")
    print(f"두 집단의 평균 판매가격 차이(약 {mean_diff:.2f}달러)는 통계적으로 유의합니다.")
else:
    print(f"\np-value({p_value:.4f}) >= alpha({alpha}) → 귀무가설 기각 실패")
    print("두 집단의 평균 판매가격 차이가 통계적으로 유의하다고 볼 수 없습니다.")

group_gd 표본 수: 20, 평균: 191916.60
group_ta 표본 수: 20, 평균: 137432.50

group_gd와 group_ta는 KitchenQual 값이 다른 서로 별개의 주택(행)에서 추출되었습니다.
한 주택이 Gd이면서 동시에 TA일 수 없고, 두 표본 사이에 짝지어진 관계도 없으므로 두 집단은 독립입니다.

Shapiro-Wilk (group_gd): 통계량=0.9555, p-value=0.4580
Shapiro-Wilk (group_ta): 통계량=0.9410, p-value=0.2503

Levene 등분산 검정: 통계량=3.7806, p-value=0.0593

등분산성 충족(p>=0.05): True -> equal_var=True

t통계량: 3.4249
p-value: 0.0015

p-value(0.0015) < alpha(0.05) → 귀무가설 기각
두 집단의 평균 판매가격 차이(약 54484.10달러)는 통계적으로 유의합니다.


### 필수 2 답변 작성란

**Q1.** 두 집단이 독립표본인 이유는 무엇인가요?  
-> gd / ta 각 집단이 서로 다른 주택으로 구성되어 하나의 관측한 값이 다른 관측값 나차의ㅜ컬ㄻ

**Q2.** 독립성은 별도의 p-value로 확인할 수 있나요?  
-> 일반적으로 독립성은 별도의 검정 =value로 확인하는 것이 아니라
-> 데이터 수집 구조, feature엔지니어링 구조등을 통해 판단

**Q3.** 정규성과 등분산성 가정은 각각 충족되나요?  
-> 겅쥬성과 등분산성 위반 근거가 없음

**Q4.** `equal_var`에는 어떤 값을 사용해야 하나요?  
-> levene p-valuse가 0.05보다 크므로 true를 사용

**Q5.** 두 집단의 평균 판매가격에는 통계적으로 유의한 차이가 있나요?
-> 유의한 차이가 있음/ t검정 결과 p-value가 약 0.0015로 0.05보다 작다.
-> 일반적으로 주방설계를 잘해놓으면 주택가격이 달라질 가능성이 있음



---

## 과제. 대응표본과 독립표본 상황 구분

### 문제 3-1. 같은 주택의 보수 전후 예상 판매가격 비교

#### 문제 설명

부동산 회사가 동일한 주택 10채에 대해 보수 전 예상 판매가격과 보수 후 예상 판매가격을 각각 산정했습니다. 같은 위치의 값은 동일한 주택의 전후 가격으로 서로 짝을 이룹니다.

```python
before_price = [145, 162, 178, 155, 190, 172, 168, 181, 159, 175]
after_price  = [154, 170, 185, 164, 201, 179, 176, 190, 168, 183]
```

단위는 천 달러입니다.

#### 요구사항

1. 두 배열의 길이가 같은지 확인하세요.
2. 보수 전후 평균을 계산하세요.
3. 독립표본 t검정과 대응표본 t검정 중 적절한 방법을 선택하고 이유를 설명하세요.
4. 대응표본 t검정의 정규성 가정은 개별 배열이 아니라 `after_price - before_price` 차이값에 적용된다는 점을 확인하세요.
5. 차이값에 Shapiro-Wilk 정규성 검정을 수행하세요.
6. 적절한 t검정을 실행하고 t통계량과 p-value를 출력하세요.
7. 보수 전후 평균 예상 판매가격에 유의한 차이가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 이 데이터는 독립표본인가요, 대응표본인가요?  
**Q2.** 대응표본 t검정에서는 무엇의 정규성을 확인해야 하나요?  
**Q3.** `stats.ttest_ind()`가 아니라 어떤 함수를 사용해야 하나요?  
**Q4.** 검정 결과 보수 전후 예상 판매가격에 유의한 차이가 있나요?

#### 제출 결과

- 표본 관계 판단과 근거
- 전후 평균과 차이값
- 차이값의 정규성 결과
- 대응표본 t검정 결과
- 결과 해석
- Q1~Q4 답변

In [3]:
# 과제 코드를 작성하세요.
import numpy as np
from scipy import stats

before_price = [145, 162, 178, 155, 190, 172, 168, 181, 159, 175]
after_price  = [154, 170, 185, 164, 201, 179, 176, 190, 168, 183]

# 1. 길이 확인
print(f"before_price 길이: {len(before_price)}")
print(f"after_price 길이: {len(after_price)}")
print(f"길이 동일 여부: {len(before_price) == len(after_price)}")

# 2. 평균 계산
before_mean = np.mean(before_price)
after_mean = np.mean(after_price)
print(f"\n보수 전 평균: {before_mean:.2f} (천 달러)")
print(f"보수 후 평균: {after_mean:.2f} (천 달러)")
print(f"평균 차이: {after_mean - before_mean:.2f} (천 달러)")

# 3. 검정 방법 선택 이유
print("\n동일한 주택 10채를 대상으로 '보수 전'과 '보수 후' 두 번 측정했으므로,")
print("두 값은 서로 짝지어진(paired) 관계입니다. 따라서 대응표본 t검정을 사용해야 합니다.")

# 4, 5. 차이값에 대한 정규성 검정
diff = np.array(after_price) - np.array(before_price)
print(f"\n차이값(after - before): {diff}")

stat_shapiro, p_shapiro = stats.shapiro(diff)
print(f"\nShapiro-Wilk (차이값): 통계량={stat_shapiro:.4f}, p-value={p_shapiro:.4f}")
print(f"차이값의 정규성 충족(p>=0.05): {p_shapiro >= 0.05}")

# 6. 대응표본 t검정 실행
t_stat, p_value = stats.ttest_rel(after_price, before_price)
print(f"\n대응표본 t검정 결과")
print(f"t통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")

# 7. 결론
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.4e}) < alpha({alpha}) → 귀무가설 기각")
    print(f"보수 전후 예상 판매가격 평균 차이(약 {after_mean-before_mean:.2f}천 달러)는 통계적으로 유의합니다.")
else:
    print(f"\np-value({p_value:.4e}) >= alpha({alpha}) → 귀무가설 기각 실패")
    print("보수 전후 예상 판매가격 차이가 통계적으로 유의하다고 볼 수 없습니다.")


before_price 길이: 10
after_price 길이: 10
길이 동일 여부: True

보수 전 평균: 168.50 (천 달러)
보수 후 평균: 177.00 (천 달러)
평균 차이: 8.50 (천 달러)

동일한 주택 10채를 대상으로 '보수 전'과 '보수 후' 두 번 측정했으므로,
두 값은 서로 짝지어진(paired) 관계입니다. 따라서 대응표본 t검정을 사용해야 합니다.

차이값(after - before): [ 9  8  7  9 11  7  8  9  9  8]

Shapiro-Wilk (차이값): 통계량=0.8871, p-value=0.1574
차이값의 정규성 충족(p>=0.05): True

대응표본 t검정 결과
t통계량: 22.8079
p-value: 0.000000

p-value(2.8420e-09) < alpha(0.05) → 귀무가설 기각
보수 전후 예상 판매가격 평균 차이(약 8.50천 달러)는 통계적으로 유의합니다.


### 과제 답변 작성란

**Q1.** 이 데이터는 독립표본인가요, 대응표본인가요?  
-> 대응표본
-> 서로 다른 주택 10채가 아닌 동일한 주택 10채의 보수 전 후를 비교하는 것이기 떄문

**Q2.** 대응표본 t검정에서는 무엇의 정규성을 확인해야 하나요?  
-> before_prica와 after_price의 차이의 정규성을 확인해야함
-> 대응표본 t검정은 차이값들의 평균이 차이가 없는지를 검정하기 때문

**Q3.** `stats.ttest_ind()`가 아니라 어떤 함수를 사용해야 하나요?  
-> 대응표본을 비교할 때 쓰는 함수인 'ttest_rel()'를 사용해야함
-> 'stats.ttest_ind()'는 독립표본을 비교할 때 쓰는 함수이기 때문

**Q4.** 검정 결과 보수 전후 예상 판매가격에 유의한 차이가 있나요?
-> 유의하다고 볼 수 있음
-> p=2.8420e-09로 0.05보다 작기 때문에 귀무가설을 기각
-> 동일한 주택의 보수 전후 판매 가격이 통계적으로 유의미한 차이가 있다는 것을 근거로 볼 수 있음


---

## 실습 마무리

1. t통계량은 어떤 두 값을 비교하여 계산하나요?
-> 관측된 평균 차이를 표준오차(그 차이의 불확실성)와 비교함

2. 단일표본 t검정은 어떤 상황에서 사용하나요?
-> 하나의 표본평균은 특정 기준값 또는 모집단 평균과 비교할 때 사용

3. 독립표본과 대응표본은 데이터 수집 구조에서 어떤 차이가 있나요?
-> 독립표본은 서로 다른 대상들로 구성되고, 대응표본은 같은 대상의 반복 측정값처럼 각 관측값이 짝을 이룸

4. 독립표본 t검정에서 정규성·등분산성·독립성은 각각 어떻게 확인하나요?
-> shapiro, levene, 독립성은 데이터 수집구조
5. 분산이 같다고 보기 어렵거나 확신할 수 없을 때 어떤 t검정을 사용할 수 있나요?
-> equla_var = False인 wehlch t검정을 사용할 수 있음